# TOQIX AI — 03 · Universal job runner (website → GPU → finished media)

Run this notebook on a **GPU runtime** (Runtime → Change runtime type → **GPU**;
a free T4 is enough), then press **Run All** and leave it open. It connects
TOQIX to its generation engines:

1. Every 30 seconds it asks the TOQIX website for waiting jobs
   (`POST /api/jobs/claim-next` — the server hands it the oldest waiting job; jobs with status
   `queued`, plus the older `exported` status, count as waiting).
2. It **atomically claims** the **oldest** waiting job (see the worker
   mesh below — only one worker can win a claim), reports **live
   progress** back to the website while it works, generates the media,
   and **uploads the finished file** to the job — the website then
   shows it as done in the gallery.

| Job kind | Engine (all open weights, no external API) | Output |
|---|---|---|
| video | Wan 2.1 T2V 1.3B (proven T4 recipe from notebook 01) | .mp4, h264 / yuv420p / faststart |
| image | SDXL-Turbo (4 steps) | .jpg |
| audio | Kokoro-82M text-to-speech (character voices) | .mp3 |

**About video speed — checked honestly (2026-10-14):** we verified the
LightX2V Hugging Face org looking for a step-distillation LoRA for
Wan2.1-T2V-**1.3B**. Their 4-step LoRAs exist only for the **14B** models
(`lightx2v/Wan2.1-Distill-Loras` — `wan2.1_t2v_14b_lora_rank64_lightx2v_4step`),
which do not fit a free T4. For the 1.3B there is no diffusers-loadable
step-distillation LoRA: `lightx2v/Wan2.1-T2V-1.3B-Distill-Models` is a single
raw `.pt` checkpoint for the LightX2V framework (not a LoRA), and the 1.3B
"longcat" repos are GRPO quality LoRAs that still need normal step counts.
So instead of inventing a repo id, video runs the base model at **14 steps**
instead of the old 30 — about **2× faster** with the exact memory recipe
that already produced TOQIX's first video on a T4. The runner always renders
video with the 1.3B model, even if a job record names the 14B (the 14B does
not fit this GPU).

Models load **lazily** — a model is only downloaded/loaded when the first
job of its kind arrives — and stay resident for later jobs. If the GPU runs
out of memory, the runner frees the other models and retries the job once.
Any job that still fails is marked **failed** on the website and the runner
moves on to the next job. To stop the runner, interrupt the last cell.

**Fast lane (2026-10-14):** video jobs now denoise small (640x368, 8 fps) and finish big in ffmpeg (motion interpolation to 16 fps + upscale to the job size) — see the video cell. First run must be timed end-to-end before any speed claim.


## One queue, many GPUs — the worker mesh

This notebook is a **worker**. Run the **same notebook** on any free
GPU — a Colab session on any Google account, a Kaggle notebook
(with **Internet ON**), a local machine — and every copy joins the
same workforce, all pulling from the one website queue:

- Each worker has a name (watch the *Worker identity* cell, e.g.
  `colab-a3f9`, `kaggle-m71q`) and takes **one job at a time**.
- Claiming is **atomic**: two workers can **never** take the same
  job. The worker that loses a claim simply moves on to the next
  waiting job.
- While a worker generates, its progress posts double as a
  **heartbeat**. If a free session dies silently, the website
  notices the missing heartbeat after **45 minutes** and returns
  the job to the queue for another worker — nothing gets stuck.
- Because the load is spread across sessions and accounts, no
  single free GPU quota burns out fast.


In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import subprocess
import sys


def _pip_install(packages, label, show_tail=False):
    """Install packages, preferring prebuilt wheels. Never raises: one
    bad package must not kill the whole worker setup.

    With show_tail=True the pip output is captured and, on failure, its
    last lines are printed at the END of the cell output: in Colab a
    long pip log scrolls away / gets truncated in the UI, which is how
    the kokoro install failure stayed invisible through three fleet
    runs (2026-10-14) -- the summary said FAILED but pip's actual error
    text was never seen."""
    print("installing", label, ":", " ".join(packages), flush=True)
    cmd = [sys.executable, "-m", "pip", "install", "--prefer-binary",
           *packages]
    if show_tail:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            tail = (result.stdout + "\n" + result.stderr).strip().splitlines()
            print("---- pip output tail for", label, "----", flush=True)
            for line in tail[-40:]:
                print(line, flush=True)
            print("---- end pip output tail ----", flush=True)
    else:
        result = subprocess.run(cmd)
    ok = result.returncode == 0
    print(label, "install", "OK" if ok else "FAILED (see output above)",
          flush=True)
    return ok


# Core packages -- all publish prebuilt wheels for Colab's Python.
_pip_install(
    ["diffusers>=0.33.0", "transformers", "accelerate", "bitsandbytes",
     "protobuf", "imageio", "imageio-ffmpeg", "soundfile", "requests"],
    "core")

# Risky packages -- installed one by one so a failure is visible and
# non-fatal (a dead install here only disables that feature; the worker
# still starts). sentencepiece is pinned to 0.2.2, the first release
# shipping prebuilt CPython 3.13 wheels: unpinned installs on Colab's
# Python 3.13 image picked a source-only release, compiled C++ for
# ~15 minutes and failed, and the old check=True aborted the whole
# setup before the worker cell ever ran (fleet attempt 1, 2026-10-14).
_pip_install(["sentencepiece==0.2.2"], "sentencepiece")
# kokoro is NOT installed into this Python 3.13 environment: kokoro
# 0.9.4 and misaki both declare requires_python <3.13, so pip can never
# resolve them here (the real cause of every "kokoro install FAILED" in
# the 2026-10-14 fleet runs). It gets its own Python 3.12 venv below,
# right after WORK_DIR exists.

# The CHAT engine (Qwen3-8B by default -- see CHAT_MODEL_ID in
# the chat cell below; loaded in 8-bit) needs no extra packages:
# transformers + bitsandbytes are already in the core group above.

print("setup finished (a FAILED line above only disables that feature)")

import gc
import os
import time
import traceback

import requests
import torch

SITE = "https://toqix.pythonanywhere.com"
# HEAVY_ONLY: when True, this worker claims ONLY heavy jobs
# (10 s+ videos) — the Kaggle runner variant sets this, because
# Kaggle's weekly GPU hours are reserved for the heavy class
# while the Colab fleet works the normal queue oldest-first.
HEAVY_ONLY = False
# /content exists only on Colab; on Kaggle (and anywhere else) work
# under the current directory instead.
WORK_DIR = ("/content/toqix_runner" if os.path.isdir("/content")
            else os.path.join(os.getcwd(), "toqix_runner"))
os.makedirs(WORK_DIR, exist_ok=True)

# --- Kokoro TTS: isolated Python 3.12 venv ------------------------------
# Built with uv (standalone CPython 3.12) because the notebook's own
# Python 3.13 can never host kokoro (see above). CPU torch is enough
# for the 82M model and keeps the download small. Never raises: if this
# fails, audio jobs fail cleanly per-job and the worker still runs.
def _setup_kokoro_venv():
    venv_dir = os.path.join(WORK_DIR, "kokoro-venv")
    venv_py = os.path.join(venv_dir, "bin", "python")
    if os.path.exists(venv_py):
        print("kokoro venv already present:", venv_dir, flush=True)
        return venv_py
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "uv"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("uv install failed: "
                               + (r.stderr or "")[-300:])
        uv = [sys.executable, "-m", "uv"]
        r = subprocess.run(uv + ["venv", "--python", "3.12", venv_dir],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("uv venv failed: " + (r.stderr or "")[-300:])
        r = subprocess.run(
            uv + ["pip", "install", "--python", venv_py, "torch",
                  "--index-url", "https://download.pytorch.org/whl/cpu"],
            capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("venv torch install failed: "
                               + (r.stderr or "")[-300:])
        r = subprocess.run(
            uv + ["pip", "install", "--python", venv_py,
                  "kokoro", "soundfile"],
            capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("venv kokoro install failed: "
                               + (r.stderr or "")[-300:])
        # misaki's English G2P needs the spaCy model en_core_web_sm, and
        # the misaki[en] extra does NOT reliably install it into a uv
        # venv -- proven live 2026-10-15: an audio job failed with
        # spaCy E050 "Can't find model 'en_core_web_sm'". Install the
        # model wheel explicitly; fall back to spaCy's own downloader.
        r = subprocess.run(
            uv + ["pip", "install", "--python", venv_py,
                  "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"],
            capture_output=True, text=True)
        if r.returncode != 0:
            r = subprocess.run([venv_py, "-m", "spacy", "download",
                                "en_core_web_sm"],
                               capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("venv spacy model install failed: "
                               + (r.stderr or "")[-300:])
        print("kokoro spacy model ready: en_core_web_sm", flush=True)
        print("kokoro venv ready:", venv_dir, flush=True)
        return venv_py
    except Exception as exc:
        print("kokoro venv setup failed (%r) -- audio jobs will fail "
              "on this worker" % (exc,), flush=True)
        return None


KOKORO_PYTHON = _setup_kokoro_venv()

# ffmpeg: prefer the binary bundled with imageio-ffmpeg (always present in
# Colab after the install above); fall back to a system ffmpeg.
try:
    import imageio_ffmpeg
    FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    FFMPEG = "ffmpeg"
print("site:", SITE)
print("ffmpeg:", FFMPEG)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def claim_next_job():
    """Ask the website for work and atomically claim the oldest
    waiting job for THIS worker. Returns the claimed job dict, or
    None when nothing is waiting (or the site could not be reached).

    The site no longer publishes its job list — creations belong to
    individual user accounts now and stay private. The server picks
    the job itself (real submissions before legacy packages, oldest
    first), skips content keys this session already handled (the
    runner's dedupe), and claims it in the same step. WORKER and
    _handled_keys are read from globals at call time: both are
    defined by later cells, before the poll loop ever runs."""
    try:
        resp = requests.post(
            SITE + "/api/jobs/claim-next",
            json={"worker": globals().get("WORKER", "unknown"),
                  "heavy_only": bool(globals().get("HEAVY_ONLY", False)),
                  "handled_keys": [list(k) for k in
                                   globals().get("_handled_keys", set())]},
            timeout=30)
        if resp.status_code == 204:
            return None
        resp.raise_for_status()
        return resp.json()
    except Exception as exc:
        print("claim-next failed (continuing):", repr(exc))
        return None


def set_progress(job_id, progress, status, error=None):
    """Report progress/status for a job. Never raises — a website hiccup
    must not kill a generation that is already running.

    Once the Worker identity cell has run, every post also carries this
    worker's name — the server treats each post as a claim HEARTBEAT
    and refreshes the job's claim stamp, so a live worker's job is
    never mistaken for a dead worker's job."""
    payload = {"progress": int(progress), "status": status}
    if error:
        payload["error"] = str(error)[:500]
    worker = globals().get("WORKER")
    if worker:
        payload["worker"] = worker
    try:
        requests.post(
            SITE + "/api/jobs/%s/progress" % job_id,
            json=payload,
            timeout=30,
        )
    except Exception as exc:
        print("progress post failed (continuing):", repr(exc))


_MIME = {".mp4": "video/mp4", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
         ".mp3": "audio/mpeg"}


def import_file(job_id, path):
    """Upload the finished media file. The server normalizes it and marks
    the job done. Returns True on success.

    BOUNDED (2026-10-15): up to 3 attempts with a 120 s timeout each
    and 20 s gaps, and network exceptions are retried like bad
    statuses. This used to be ONE 600 s attempt whose exceptions
    escaped as a bare "output import to the site failed", and whose
    hang could pin a worker invisibly. Never raises: the caller
    posts the job failed (with the real error) on a False."""
    ext = os.path.splitext(path)[1].lower()
    mime = _MIME.get(ext, "application/octet-stream")
    for attempt in (1, 2, 3):
        try:
            with open(path, "rb") as fh:
                resp = requests.post(
                    SITE + "/api/jobs/%s/import" % job_id,
                    files={"file": (os.path.basename(path), fh, mime)},
                    timeout=120,
                )
            print("import", job_id, "-> HTTP", resp.status_code,
                  resp.text[:200])
            if resp.ok:
                return True
            print("import attempt %d: HTTP %s (will retry)"
                  % (attempt, resp.status_code))
        except Exception as exc:
            print("import attempt %d failed: %r" % (attempt, exc))
        if attempt < 3:
            time.sleep(20)
    return False


def is_cuda_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower())


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ------------------------------------------------------------------
# Worker identity — this notebook is ONE worker in the TOQIX mesh.
#
# Run the SAME notebook on any free GPU (Colab on any account, Kaggle
# with Internet ON, a local machine) and every copy works the SAME
# website queue in parallel. A worker takes ONE job at a time, and
# the website's atomic claim means two workers can never grab the
# same job. If this session dies mid-job, the claim goes stale after
# 45 minutes without a progress heartbeat and the job returns to
# the queue for another worker.
# ------------------------------------------------------------------
import random
import string


def _detect_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if os.environ.get("COLAB_GPU") or os.environ.get("COLAB_TPU_ADDR"):
        return "colab"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except Exception:
        pass
    return "local"


_suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=4))
WORKER = "%s-%s" % (_detect_platform(), _suffix)
print("worker identity:", WORKER)


def claim_job(job_id):
    """Atomically claim a waiting job for THIS worker.

    Returns (True, job_dict) when this worker won the claim, and
    (False, None) when another worker already holds the job (the
    website answers 409) or the claim could not be posted. Never
    raises — the loop simply moves on to the next waiting job."""
    try:
        resp = requests.post(
            SITE + "/api/jobs/%s/claim" % job_id,
            json={"worker": WORKER},
            timeout=30,
        )
        if resp.status_code == 200:
            return True, resp.json()
        if resp.status_code == 409:
            return False, None
        print("claim %s -> HTTP %s (treated as not claimable)"
              % (job_id, resp.status_code))
        return False, None
    except Exception as exc:
        print("claim post failed (continuing):", repr(exc))
        return False, None


## Video engine — Wan 2.1 T2V 1.3B (proven T4 recipe + 14-step speed layer)

Same memory recipe that generated TOQIX's first video on a free T4
(notebook 01): the UMT5-XXL text encoder loads in **8-bit** (bitsandbytes)
pinned to the GPU — a bf16 encoder is ~11 GB and OOM-crashes the ~12.7 GB
free-Colab system RAM before offload even starts. Each job's prompt is
encoded **once**, then the encoder is stripped from the pipeline before
denoising (it is re-loaded from the local cache for the next job — a small
cost that keeps the proven memory profile for every job). Denoising runs
with **sequential CPU offload**, transformer + embeddings in **float16**
(the T4/Turing has no bf16 attention kernels — efficient SDPA accepts
Half/Float only) and the native-efficient attention backend (the math path
materializes an ~18 GiB attention matrix and VRAM-OOMs the T4). The VAE
stays fp32 with tiling + slicing.

Speed layer: **14 inference steps, guidance 5.0** (see the intro note for
why no distilled LoRA is used), live per-step progress (5% → 95%), frame
count from the job's duration at 16 fps (capped at 121 frames), size from
the job's aspect ratio, and a final ffmpeg pass to h264 / yuv420p /
limited-range BT.709 / faststart.


In [ ]:
from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video
from transformers import BitsAndBytesConfig, UMT5EncoderModel

VIDEO_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
VIDEO_STEPS = 14          # speed layer: 14 steps instead of the old 30
VIDEO_GUIDANCE = 5.0
VIDEO_FPS = 16
VIDEO_SIZES = {"16:9": (832, 480), "9:16": (480, 832), "1:1": (640, 640)}
# FAST LANE (generate-small-finish-big, added 2026-10-14): denoise at a
# smaller size and HALF the frame rate, then let ffmpeg finish big --
# motion interpolation back to 16 fps + lanczos upscale to the job size.
# Denoising cost scales ~ frames x pixels, so this cuts it to roughly
# 0.5 x 0.58 = ~0.29x of the full-size 14-step run (~3.4x faster).
# Set FAST_VIDEO = False to return to full-size generation.
FAST_VIDEO = True
FAST_SIZES = {"16:9": (640, 368), "9:16": (368, 640), "1:1": (512, 512)}
FAST_GEN_FPS = 8          # frames generated at 8 fps, finished to 16
# Frame counts come from the job's requested duration: the fast
# lane generates duration x 8 frames (+1 keeps Wan's 4k+1 form)
# and the ffmpeg finish below interpolates to duration x 16
# frames at 16 fps — a 10 s job (the site's HEAVY class)
# generates 81 frames and finishes as 161. The MAX_* ceilings
# only guard durations the site does not offer.
MAX_GEN_FRAMES = 121      # fast-lane ceiling (15 s at 8 fps)
MAX_FRAMES = 161          # full-size ceiling (10 s at 16 fps)
DEFAULT_NEGATIVE = (
    "low quality, worst quality, blurry, distorted, deformed, extra limbs, "
    "flickering, jittery motion, static frame, watermark, text, subtitles")

_video = {"pipe": None, "offload_ready": False}


def _load_video_encoder():
    """Fresh 8-bit UMT5 encoder, pinned to the GPU (~5.7 GB, not ~11 GB)."""
    return UMT5EncoderModel.from_pretrained(
        VIDEO_MODEL_ID, subfolder="text_encoder",
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map={"": "cuda"}, torch_dtype=torch.bfloat16)


def ensure_video_pipe():
    if _video["pipe"] is not None:
        return _video["pipe"]
    print("loading Wan 2.1 video pipeline (first video job)...")
    transformer = WanTransformer3DModel.from_pretrained(
        VIDEO_MODEL_ID, subfolder="transformer", torch_dtype=torch.float16)
    vae = AutoencoderKLWan.from_pretrained(
        VIDEO_MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    encoder = _load_video_encoder()
    pipe = WanPipeline.from_pretrained(
        VIDEO_MODEL_ID, text_encoder=encoder, transformer=transformer,
        vae=vae, torch_dtype=torch.float16)
    try:
        pipe.vae.enable_tiling()
        pipe.vae.enable_slicing()
    except Exception:
        pass
    _video["pipe"] = pipe
    print("video pipeline ready")
    return pipe


def free_video():
    if _video["pipe"] is not None:
        del _video["pipe"]
        _video["pipe"] = None
        _video["offload_ready"] = False
    free_cuda()


def _render_clip(job, prompt, seconds, tag):
    """Denoise ONE clip for `prompt` at `seconds` and run the fast-
    lane finish; returns the finished .mp4 path. `tag` suffixes the
    working filenames ("" for a normal job, "_s1"/"_s2" for scenes).

    This is the runner's original single-clip path, parameterised:
    normal jobs call it once; heavy scene-structured jobs call it
    once per scene with that scene's own prompt."""
    jid = job["id"]
    stem = jid + tag
    set_progress(jid, 2, "generating")  # heartbeat: model load starting
    pipe = ensure_video_pipe()
    set_progress(jid, 4, "generating")  # model ready

    duration = int(round(float(seconds)))
    target_w, target_h = VIDEO_SIZES.get(job.get("aspect") or "16:9",
                                         VIDEO_SIZES["16:9"])
    if FAST_VIDEO:
        gen_fps = FAST_GEN_FPS
        num_frames = min(duration * gen_fps + 1, MAX_GEN_FRAMES)
        width, height = FAST_SIZES.get(job.get("aspect") or "16:9",
                                       FAST_SIZES["16:9"])
        print("FAST LANE: %ds job%s — generating %d frames at %dx%d / "
              "%d fps, finishing to %dx%d / %d fps"
              % (duration, " (HEAVY)" if job.get("heavy") else "",
                 num_frames, width, height, gen_fps,
                 target_w, target_h, VIDEO_FPS))
    else:
        gen_fps = VIDEO_FPS
        num_frames = min(duration * VIDEO_FPS + 1, MAX_FRAMES)
        width, height = target_w, target_h
    try:
        seed = int(job.get("seed", 42))
    except Exception:
        seed = 42
    negative = job.get("negative_prompt") or DEFAULT_NEGATIVE

    # Encode this job's prompt ONCE with the GPU-resident 8-bit encoder,
    # then strip the encoder from the pipeline before denoising — the
    # single biggest RAM resident, and denoising only needs the embeddings.
    encoder = pipe.text_encoder
    if encoder is None:
        encoder = _load_video_encoder()
        pipe.text_encoder = encoder
    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=prompt, negative_prompt=negative,
        do_classifier_free_guidance=True, device="cuda")
    pipe.text_encoder = None
    del encoder
    gc.collect()
    torch.cuda.empty_cache()

    # Offload + attention backend are set up AFTER the strip (once), so the
    # offload hooks never re-stage the encoder onto the CPU.
    if not _video["offload_ready"]:
        try:
            pipe.enable_sequential_cpu_offload()
        except Exception:
            pipe.enable_model_cpu_offload()
        try:
            pipe.transformer.set_attention_backend("_native_efficient")
        except Exception:
            pass
        _video["offload_ready"] = True

    def on_step_end(pipe_, step_index, timestep, callback_kwargs):
        pct = 5 + int(90 * (step_index + 1) / VIDEO_STEPS)
        set_progress(jid, min(pct, 95), "generating")
        return callback_kwargs

    generator = torch.Generator(device="cpu").manual_seed(seed)
    frames = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        height=height, width=width, num_frames=num_frames,
        guidance_scale=VIDEO_GUIDANCE, num_inference_steps=VIDEO_STEPS,
        generator=generator, callback_on_step_end=on_step_end,
    ).frames[0]

    raw_path = os.path.join(WORK_DIR, stem + "_raw.mp4")
    export_to_video(frames, raw_path, fps=gen_fps)

    # Delivery encode: h264 / yuv420p / limited-range BT.709 / faststart.
    final_path = os.path.join(WORK_DIR, stem + ".mp4")
    cmd = [FFMPEG, "-y", "-i", raw_path,
           "-c:v", "libx264", "-pix_fmt", "yuv420p",
           "-colorspace", "bt709", "-color_primaries", "bt709",
           "-color_trc", "bt709", "-color_range", "tv",
           "-movflags", "+faststart", final_path]
    if FAST_VIDEO:
        # Finish big: interpolate 8 -> 16 fps and upscale to the job size
        # BEFORE the encoder settings (cmd[4] is right after the input).
        cmd[4:4] = ["-vf",
                    "minterpolate=fps=%d:mi_mode=mci:mc_mode=aobmc:"
                    "me_mode=bidir,scale=%d:%d:flags=lanczos"
                    % (VIDEO_FPS, target_w, target_h)]
    result = subprocess.run(cmd, capture_output=True, text=True,
                            timeout=1200)
    if result.returncode != 0 or not os.path.exists(final_path):
        print("ffmpeg re-encode failed, importing the raw export instead:",
              result.stderr[-500:] if result.stderr else "")
        return raw_path
    return final_path


def _concat_clips(job, clip_paths):
    """Join finished scene clips into ONE mp4 at the job's target
    size / 16 fps, same delivery spec as a single clip (h264,
    yuv420p, limited BT.709, faststart). Each input is normalised
    (fps + scale + SAR) before the concat, so clips always join
    cleanly. On any failure, returns the first clip — a heavy job
    still delivers something rather than failing outright."""
    jid = job["id"]
    target_w, target_h = VIDEO_SIZES.get(job.get("aspect") or "16:9",
                                         VIDEO_SIZES["16:9"])
    out_path = os.path.join(WORK_DIR, jid + "_joined.mp4")
    cmd = [FFMPEG, "-y"]
    for path in clip_paths:
        cmd += ["-i", path]
    filters = []
    for i in range(len(clip_paths)):
        filters.append(
            "[%d:v]fps=%d,scale=%d:%d:flags=lanczos,setsar=1[v%d]"
            % (i, VIDEO_FPS, target_w, target_h, i))
    filters.append(
        "".join("[v%d]" % i for i in range(len(clip_paths)))
        + "concat=n=%d:v=1:a=0[out]" % len(clip_paths))
    cmd += ["-filter_complex", ";".join(filters), "-map", "[out]",
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-colorspace", "bt709", "-color_primaries", "bt709",
            "-color_trc", "bt709", "-color_range", "tv",
            "-movflags", "+faststart", out_path]
    result = subprocess.run(cmd, capture_output=True, text=True,
                            timeout=900)
    if result.returncode != 0 or not os.path.exists(out_path):
        print("scene concat failed, delivering the first scene only:",
              result.stderr[-400:] if result.stderr else "")
        return clip_paths[0]
    return out_path


def generate_video(job):
    """Generate one video job; returns the final .mp4 path.

    Heavy jobs whose package carries `scenes` (2+ scene prompts
    from the site's prompt engine) render one clip per scene —
    ceil(duration / scenes) seconds each, same seed, the character
    anchor already locked verbatim inside each scene prompt — then
    concatenate. Every video then passes through the sound stage:
    a generated MusicGen sound bed muxed on as an AAC track. The
    sound stage is failure-safe: on ANY error the video-only file
    is returned and the job still completes."""
    jid = job["id"]
    try:
        duration = int(round(float(job.get("duration") or 5)))
    except Exception:
        duration = 5
    scenes = [s for s in (job.get("scenes") or []) if s and s.strip()]
    if len(scenes) >= 2:
        per_scene = max(2, -(-duration // len(scenes)))
        print("HEAVY scene structure: %d scenes x ~%ds (same seed, "
              "locked character anchor)" % (len(scenes), per_scene))
        clip_paths = []
        for i, scene_prompt in enumerate(scenes):
            print("scene %d/%d: %s"
                  % (i + 1, len(scenes), scene_prompt[:100]))
            clip_paths.append(
                _render_clip(job, scene_prompt, per_scene,
                             "_s%d" % (i + 1)))
        video_path = _concat_clips(job, clip_paths)
    else:
        prompt = job.get("enhanced_prompt") or job.get("prompt") or ""
        video_path = _render_clip(job, prompt, duration, "")
    # Stage breadcrumb 96: render + concat + encode are DONE. Every
    # step after this line was a SILENT zone in the old code -- no
    # progress posts at all between the last denoise step (95) and
    # the import -- so a worker could hang there invisibly until
    # its claim went stale (job dd92666f4b98, 2026-10-15). Each
    # stage boundary below now posts its own milestone.
    set_progress(jid, 96, "generating")
    return add_sound_stage(job, video_path, duration)


# ------------------------------------------------------------------
# SOUND STAGE — every video gets a generated sound bed (2026-10-15).
#
# Wan renders silent video; the owner's first 10s site video arrived
# with NO audio track at all. After rendering, this stage generates
# a music/ambience bed with facebook/musicgen-small (transformers,
# fp16 on CUDA), matched to the job's `sound_prompt` (built by the
# site's prompt engine; a local fallback covers old packages), and
# muxes it onto the MP4 with ffmpeg — video stream copied untouched,
# audio AAC 192k stereo.
#
# FAILURE-SAFE BY DESIGN: the whole stage is wrapped so ANY error
# (model download, load, generation, mux) logs "[audio] skipped: ..."
# and returns the video-only file. Audio can never fail a job.
#
# The model runs in a CHILD PROCESS (see the music bridge below),
# only after the Wan pipeline is freed in the parent, and its
# weights cache under WORK_DIR like the other fleet models.
# ------------------------------------------------------------------

MUSIC_MODEL_ID = "facebook/musicgen-small"
MUSIC_MAX_SECONDS = 30.0    # token budget ceiling: 50 tokens/second

_music = {"model": None, "processor": None}


def ensure_music_pipe():
    if _music["model"] is not None:
        return _music["model"], _music["processor"]
    from transformers import (AutoProcessor,
                              MusicgenForConditionalGeneration)
    print("loading MusicGen sound model (first video job)...")
    cache_dir = os.path.join(WORK_DIR, "hf_cache")
    model = MusicgenForConditionalGeneration.from_pretrained(
        MUSIC_MODEL_ID, torch_dtype=torch.float16, cache_dir=cache_dir)
    model.to("cuda")
    processor = AutoProcessor.from_pretrained(
        MUSIC_MODEL_ID, cache_dir=cache_dir)
    _music["model"] = model
    _music["processor"] = processor
    print("MusicGen ready")
    return model, processor


def free_music():
    if _music["model"] is not None:
        del _music["model"]
        _music["model"] = None
    _music["processor"] = None
    free_cuda()


def _fallback_sound_prompt(job):
    """Sound-bed text for job packages that predate `sound_prompt`
    (or carry an empty one) — domain/keyword derived, runner-side,
    mirroring the site prompt engine's table in miniature."""
    text = ((job.get("prompt") or "") + " "
            + (job.get("domain") or "")).lower()
    for keys, bed in (
        (("fight", "battle", "war", "sports", "action"),
         "epic cinematic orchestral battle music, heavy drums, "
         "brass, intense and heroic"),
        (("ocean", "sea", "wave", "beach", "ocean_weather"),
         "calm cinematic ambient music with gentle ocean waves, "
         "soft piano, peaceful and spacious"),
        (("space", "planet", "rocket", "astronaut", "space_scifi"),
         "dark cinematic synth ambient music, deep pads and slow "
         "pulses, vast and mysterious"),
        (("city", "street", "urban", "cities_streets"),
         "modern urban ambience, electronic pulse, subtle city "
         "sounds, energetic and stylish"),
        (("forest", "nature", "panda", "garden", "animals",
          "nature_landscapes"),
         "gentle peaceful ambient music, soft piano and nature "
         "sounds, birds chirping"),
    ):
        if any(k in text for k in keys):
            return bed
    return "cinematic ambient music matching the scene mood"


def _generate_sound_bed(model, processor, prompt, duration, wav_path):
    """Run MusicGen for ~`duration` seconds (capped) -> stereo-ready
    WAV on disk. MusicGen emits 50 codec tokens per second of audio;
    the model is mono, the mux stage upmixes to stereo."""
    import soundfile as sf

    seconds = min(max(float(duration or 3), 1.0), MUSIC_MAX_SECONDS)
    inputs = processor(text=[prompt], padding=True, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        audio_values = model.generate(
            **inputs, do_sample=True, guidance_scale=3.0,
            max_new_tokens=int(seconds * 50))
    sample_rate = model.config.audio_encoder.sampling_rate
    data = audio_values[0].float().cpu().numpy()  # (channels, samples)
    sf.write(wav_path, data.T, sample_rate)
    return wav_path


def mux_audio(video_path, wav_path, out_path):
    """Mux a WAV sound bed onto an MP4: video stream COPIED (the
    finished encode is never re-compressed), audio AAC 192k stereo,
    faststart kept. Returns True on success.

    Duration rule: the VIDEO length always wins. `apad` pads a
    short bed with silence and `-shortest` then ends the file at
    the video's end (trimming a long bed) — without apad, a bed
    even slightly shorter than the clip would truncate the video
    itself (proven in the local mux test)."""
    cmd = [FFMPEG, "-y", "-i", video_path, "-i", wav_path,
           "-map", "0:v:0", "-map", "1:a:0",
           "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-ac", "2",
           "-af", "apad", "-shortest",
           "-movflags", "+faststart", out_path]
    # timeout=300: a hung ffmpeg here wedged workers at
    # breadcrumb 97 forever (jobs 1bf3669ee8ea / e1aa8751615a,
    # 2026-10-15). TimeoutExpired propagates to add_sound_stage's
    # catch-all -> "[audio] skipped" -> video-only delivery.
    result = subprocess.run(cmd, capture_output=True, text=True,
                            timeout=300)
    if result.returncode != 0 or not os.path.exists(out_path):
        print("audio mux failed:",
              result.stderr[-400:] if result.stderr else "")
        return False
    return True


# ------------------------------------------------------------------
# MusicGen runs in a CHILD PROCESS (2026-10-15), mirroring the
# Kokoro bridge pattern: a small script on disk, one subprocess per
# job, a JSON request file, captured output.
#
# WHY: the in-process stage is the prime suspect in the
# dd92666f4b98 freeze. After the Wan pipeline (with its sequential
# CPU-offload hooks) was freed in-process, the MusicGen load /
# generate could hang the worker's main thread indefinitely: no
# exception, no progress post, the heartbeat thread dead with the
# process, the claim stale, the job re-claimed, Kaggle quota
# burned -- in a loop. A child process can be KILLED: subprocess
# timeout 420 s -> the parent logs "[audio] skipped: stage timed
# out" and delivers the video-only file. The sound stage can now
# never hang or kill a job. (ensure_music_pipe / _generate_sound_
# bed above are retained for reference but no longer used.)
# ------------------------------------------------------------------
import json

MUSIC_BRIDGE_TIMEOUT = 420  # seconds; a bed takes ~1-3 min on a T4

_MUSIC_BRIDGE_SRC = """
import json
import sys

import soundfile as sf
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

req = json.load(open(sys.argv[1]))
model = MusicgenForConditionalGeneration.from_pretrained(
    req["model_id"], torch_dtype=torch.float16,
    cache_dir=req["cache_dir"])
model.to("cuda")
processor = AutoProcessor.from_pretrained(
    req["model_id"], cache_dir=req["cache_dir"])
seconds = min(max(float(req["duration"]), 1.0),
              float(req["max_seconds"]))
inputs = processor(text=[req["prompt"]], padding=True,
                   return_tensors="pt")
inputs = {k: v.to("cuda") for k, v in inputs.items()}
with torch.no_grad():
    audio_values = model.generate(
        **inputs, do_sample=True, guidance_scale=3.0,
        max_new_tokens=int(seconds * 50))
sample_rate = model.config.audio_encoder.sampling_rate
data = audio_values[0].float().cpu().numpy()  # (channels, samples)
sf.write(req["out_wav"], data.T, sample_rate)
print("music bridge wrote", req["out_wav"])
"""

MUSIC_BRIDGE_PATH = os.path.join(WORK_DIR, "music_bridge.py")
try:
    with open(MUSIC_BRIDGE_PATH, "w") as _fh:
        _fh.write(_MUSIC_BRIDGE_SRC)
except Exception as _exc:
    print("could not write music bridge (%r)" % (_exc,))


def _run_music_bridge(prompt, duration, wav_path):
    """Generate the sound bed in a child process with a hard
    timeout. Returns True when the WAV exists and the child exited
    cleanly; False (the caller delivers the video-only file) on
    timeout, crash or nonzero exit."""
    req_path = os.path.join(WORK_DIR, "music_req.json")
    try:
        with open(req_path, "w") as fh:
            json.dump(
                {"prompt": prompt,
                 "duration": float(duration or 3),
                 "max_seconds": MUSIC_MAX_SECONDS,
                 "out_wav": wav_path,
                 "model_id": MUSIC_MODEL_ID,
                 "cache_dir": os.path.join(WORK_DIR, "hf_cache")},
                fh)
        result = subprocess.run(
            [sys.executable, MUSIC_BRIDGE_PATH, req_path],
            capture_output=True, text=True,
            timeout=MUSIC_BRIDGE_TIMEOUT)
    except subprocess.TimeoutExpired:
        print("[audio] skipped: stage timed out after %ds"
              % MUSIC_BRIDGE_TIMEOUT)
        return False
    except Exception as exc:
        print("[audio] skipped: stage could not run: %r" % (exc,))
        return False
    if result.returncode != 0 or not os.path.exists(wav_path):
        tail = ((result.stdout or "") + "\n" + (result.stderr or "")
                ).strip().splitlines()
        print("[audio] skipped: stage died (rc=%s): %s"
              % (result.returncode, " | ".join(tail[-5:])))
        return False
    print("[audio] sound bed ready:", wav_path)
    return True


def add_sound_stage(job, video_path, duration):
    """Attach a generated sound bed to a finished video. NEVER
    raises and NEVER fails the job: any problem logs "[audio]
    skipped: ..." and returns the original video-only path.

    Stage breadcrumbs: 97 posts right before the MusicGen stage
    starts (Wan already freed), 98 posts when the stage is over --
    bed muxed OR stage skipped -- so the site always shows exactly
    where a video job is."""
    jid = job["id"]
    if not video_path or not os.path.exists(video_path):
        print("[audio] skipped: no video file to attach sound to")
        return video_path
    result = video_path
    try:
        print("[audio] generating sound bed")
        # Wan stays RESIDENT for the next video job (speed build,
        # 2026-10-15): with sequential CPU offload its submodules
        # park on the CPU at rest, so the music child still gets
        # the card in practice -- only the CUDA cache is dropped.
        # The old behaviour (free Wan here, reload it next job)
        # made EVERY video job pay a full pipeline reload. If the
        # child ever cannot fit it fails safe ("[audio] skipped")
        # and the video-only file is delivered.
        free_cuda()
        set_progress(jid, 97, "generating")
        prompt = (job.get("sound_prompt") or "").strip() \
            or _fallback_sound_prompt(job)
        print("[audio] sound prompt: %s" % prompt)
        wav_path = os.path.join(WORK_DIR, jid + "_sound.wav")
        if _run_music_bridge(prompt, duration, wav_path):
            out_path = os.path.join(WORK_DIR, jid + "_av.mp4")
            if mux_audio(video_path, wav_path, out_path):
                print("[audio] muxed OK")
                result = out_path
            else:
                print("[audio] skipped: mux failed")
    except Exception as exc:
        print("[audio] skipped: %r" % (exc,))
        result = video_path
    # Breadcrumb 98: the sound stage is over, one way or the other;
    # the only step left after this is the import (99, in the loop).
    set_progress(jid, 98, "generating")
    return result


## Image engine — SDXL-Turbo (4 steps)

`stabilityai/sdxl-turbo` in float16: 4 inference steps, guidance 0.0 (that
is how Turbo is designed to run). The picture is saved as PNG, converted to
an RGB **JPG** (quality 90), and the JPG is what gets imported.


In [ ]:
from diffusers import AutoPipelineForText2Image
from PIL import Image

IMAGE_MODEL_ID = "stabilityai/sdxl-turbo"
IMAGE_SIZES = {"1:1": (1024, 1024), "16:9": (1216, 704), "9:16": (704, 1216)}

_image = {"pipe": None}


def ensure_image_pipe():
    if _image["pipe"] is not None:
        return _image["pipe"]
    print("loading SDXL-Turbo image pipeline (first image job)...")
    pipe = AutoPipelineForText2Image.from_pretrained(
        IMAGE_MODEL_ID, torch_dtype=torch.float16, variant="fp16")
    pipe.to("cuda")
    _image["pipe"] = pipe
    print("image pipeline ready")
    return pipe


def free_image():
    if _image["pipe"] is not None:
        del _image["pipe"]
        _image["pipe"] = None
    free_cuda()


def generate_image(job):
    """Generate one image job; returns the final .jpg path."""
    jid = job["id"]
    pipe = ensure_image_pipe()
    width, height = IMAGE_SIZES.get(job.get("aspect") or "1:1",
                                    IMAGE_SIZES["1:1"])
    try:
        seed = int(job.get("seed", 42))
    except Exception:
        seed = 42
    prompt = job.get("enhanced_prompt") or job.get("prompt") or ""

    set_progress(jid, 10, "generating")
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(prompt=prompt, width=width, height=height,
                 num_inference_steps=4, guidance_scale=0.0,
                 generator=generator).images[0]
    set_progress(jid, 60, "generating")

    png_path = os.path.join(WORK_DIR, jid + ".png")
    jpg_path = os.path.join(WORK_DIR, jid + ".jpg")
    image.save(png_path)
    Image.open(png_path).convert("RGB").save(jpg_path, quality=90)
    set_progress(jid, 95, "generating")
    return jpg_path


## Audio engine — Kokoro-82M text-to-speech (character voices)

The job's **style** names the character; the runner maps it to a Kokoro
voice and speaks the job's **original prompt text** (`job["prompt"]`).
British voices (Emma, George) use Kokoro's British pipeline (`lang_code
"b"`), the others the American one (`"a"`). If the chosen voice fails to
load, it falls back to `af_heart`. Chunks are joined into one WAV
(soundfile) and converted to MP3 for import.


In [ ]:
import json as _json

import numpy as np
import soundfile as sf

# Kokoro cannot be imported here (Python 3.13 vs its <3.13 pin -- see
# the setup cell). Synthesis runs in the isolated 3.12 venv through a
# bridge script: one subprocess per call, so no state leaks between
# jobs and a crash fails only that one job.
_BRIDGE_SRC = """
import json
import sys

import numpy as np
import soundfile as sf
from kokoro import KPipeline

req = json.load(open(sys.argv[1]))
text, voice, out_wav = req["text"], req["voice"], req["out"]
lang_code = "b" if voice.startswith("b") else "a"
pipeline = KPipeline(lang_code=lang_code)
chunks = []
for _g, _p, audio in pipeline(text, voice=voice, speed=1.0):
    if audio is not None:
        chunks.append(np.asarray(audio, dtype=np.float32))
if not chunks:
    raise SystemExit("kokoro produced no audio")
sf.write(out_wav, np.concatenate(chunks), 24000)
print("bridge wrote", out_wav)
"""

BRIDGE_PATH = os.path.join(WORK_DIR, "kokoro_bridge.py")
try:
    with open(BRIDGE_PATH, "w") as _fh:
        _fh.write(_BRIDGE_SRC)
except Exception as _exc:
    print("could not write kokoro bridge (%r)" % (_exc,))

_kokoro_python = globals().get("KOKORO_PYTHON")
if _kokoro_python:
    print("kokoro venv python:", _kokoro_python)
else:
    print("kokoro unavailable (no 3.12 venv) -- audio jobs will fail "
          "on this worker")

KOKORO_SR = 24000
# Character name (job["style"]) -> Kokoro voice id.
VOICE_MAP = {
    "ava": "af_heart",
    "emma": "bf_emma",
    "michael": "am_michael",
    "george": "bm_george",
    "sky": "af_sky",
    "adam": "am_adam",
}
DEFAULT_VOICE = "af_heart"

def free_audio():
    free_cuda()


def _speak_one_shot(text, voice):
    """Synthesize text with one voice via the ONE-SHOT bridge: a
    fresh subprocess + a fresh KPipeline for this one call. This is
    the original audio path, kept as the fallback for when the
    resident server (below) is unavailable."""
    if not _kokoro_python:
        raise RuntimeError("kokoro is not installed on this worker")
    req_path = os.path.join(WORK_DIR, "kokoro_req.json")
    out_wav = os.path.join(WORK_DIR, "kokoro_out.wav")
    with open(req_path, "w") as fh:
        _json.dump({"text": text, "voice": voice, "out": out_wav}, fh)
    result = subprocess.run(
        [_kokoro_python, BRIDGE_PATH, req_path],
        capture_output=True, text=True, timeout=1200)
    if result.returncode != 0 or not os.path.exists(out_wav):
        tail = (result.stdout + "\n" + result.stderr).strip().splitlines()
        raise RuntimeError("kokoro bridge failed: "
                           + " | ".join(tail[-5:]))
    samples, _sr = sf.read(out_wav, dtype="float32")
    return np.asarray(samples, dtype=np.float32)


# ------------------------------------------------------------------
# RESIDENT KOKORO SERVER (speed build, 2026-10-15).
#
# The one-shot bridge above pays the full engine start on EVERY
# audio job: venv interpreter boot + torch/kokoro/spaCy imports + a
# fresh KPipeline (model load). On an ephemeral worker that was
# most of an audio job's wall time. The server below is a
# long-lived child (same venv python) that loads Kokoro ONCE --
# both English pipelines, 'a' and 'b' -- and then serves requests
# as JSON lines on stdin/stdout, writing each WAV to the path in
# the request. _speak prefers the server; ANY failure (won't start,
# dies mid-request, times out, answers ok=False) falls back to the
# one-shot bridge for that call and kills the server, so the next
# job starts a fresh one. Timeout discipline mirrors the one-shot
# path: 1200 s per request.
# ------------------------------------------------------------------
import queue as _queue
import threading as _threading

_SERVER_SRC = """
import json
import sys

import numpy as np
import soundfile as sf
from kokoro import KPipeline

_pipes = {}


def _pipe_for(voice):
    lang = "b" if voice.startswith("b") else "a"
    if lang not in _pipes:
        _pipes[lang] = KPipeline(lang_code=lang)
    return _pipes[lang]


# Preload both English variants so the first real request is fast;
# a variant that fails here is retried lazily on its first request.
for _lang in ("a", "b"):
    try:
        _pipes[_lang] = KPipeline(lang_code=_lang)
    except Exception:
        pass

print(json.dumps({"ready": True}), flush=True)
for _line in sys.stdin:
    _line = _line.strip()
    if not _line:
        continue
    _req = None
    try:
        _req = json.loads(_line)
        _chunks = []
        for _g, _p, _audio in _pipe_for(_req["voice"])(
                _req["text"], voice=_req["voice"], speed=1.0):
            if _audio is not None:
                _chunks.append(np.asarray(_audio, dtype=np.float32))
        if not _chunks:
            raise RuntimeError("kokoro produced no audio")
        sf.write(_req["out"], np.concatenate(_chunks), 24000)
        print(json.dumps({"id": _req.get("id"), "ok": True,
                          "out": _req["out"]}), flush=True)
    except Exception as _exc:
        print(json.dumps(
            {"id": (_req or {}).get("id"), "ok": False,
             "error": "%s: %s" % (type(_exc).__name__, _exc)}),
            flush=True)
"""

SERVER_PATH = os.path.join(WORK_DIR, "kokoro_server.py")
try:
    with open(SERVER_PATH, "w") as _fh:
        _fh.write(_SERVER_SRC)
except Exception as _exc:
    print("could not write kokoro server (%r)" % (_exc,))

KOKORO_SERVER_START_TIMEOUT = 600   # imports + two pipeline loads
KOKORO_SPEAK_TIMEOUT = 1200         # same as the one-shot bridge

_kokoro_server = {"proc": None, "queue": None, "seq": 0}


def _stop_kokoro_server():
    proc = _kokoro_server["proc"]
    _kokoro_server["proc"] = None
    _kokoro_server["queue"] = None
    if proc is not None:
        try:
            proc.kill()
        except Exception:
            pass


def _start_kokoro_server():
    """Ensure the resident server is running and answered ready.
    Returns True on success; on any failure the caller falls back
    to the one-shot bridge."""
    proc = _kokoro_server["proc"]
    if (proc is not None and proc.poll() is None
            and _kokoro_server["queue"] is not None):
        return True
    _stop_kokoro_server()
    if not _kokoro_python:
        return False
    try:
        log = open(os.path.join(WORK_DIR, "kokoro_server.log"), "ab")
        proc = subprocess.Popen(
            [_kokoro_python, SERVER_PATH],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=log, text=True, bufsize=1)
    except Exception as exc:
        print("kokoro server could not start: %r" % (exc,))
        return False
    lines = _queue.Queue()

    def _reader():
        try:
            for line in proc.stdout:
                try:
                    lines.put(_json.loads(line))
                except Exception:
                    pass  # stray prints must not kill the channel
        except Exception:
            pass

    _threading.Thread(target=_reader, daemon=True).start()
    _kokoro_server["proc"] = proc
    _kokoro_server["queue"] = lines
    deadline = time.time() + KOKORO_SERVER_START_TIMEOUT
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        try:
            msg = lines.get(timeout=5)
        except _queue.Empty:
            continue
        if msg.get("ready"):
            print("kokoro server ready (pid %s)" % proc.pid)
            return True
    print("kokoro server did not become ready -- one-shot bridge "
          "will be used for this job")
    _stop_kokoro_server()
    return False


def _speak_via_server(text, voice):
    """One request/response round-trip with the resident server."""
    if not _start_kokoro_server():
        raise RuntimeError("kokoro server unavailable")
    _kokoro_server["seq"] += 1
    rid = _kokoro_server["seq"]
    proc = _kokoro_server["proc"]
    lines = _kokoro_server["queue"]
    out_wav = os.path.join(WORK_DIR, "kokoro_srv_%d.wav" % rid)
    proc.stdin.write(_json.dumps(
        {"id": rid, "text": text, "voice": voice,
         "out": out_wav}) + "\n")
    proc.stdin.flush()
    deadline = time.time() + KOKORO_SPEAK_TIMEOUT
    while time.time() < deadline:
        try:
            msg = lines.get(timeout=10)
        except _queue.Empty:
            if proc.poll() is not None:
                raise RuntimeError("kokoro server exited mid-request")
            continue
        if msg.get("id") != rid:
            continue
        if not msg.get("ok"):
            raise RuntimeError("kokoro server error: %s"
                               % msg.get("error"))
        samples, _sr = sf.read(out_wav, dtype="float32")
        return np.asarray(samples, dtype=np.float32)
    raise RuntimeError("kokoro server request timed out after %ds"
                       % KOKORO_SPEAK_TIMEOUT)


def _speak(text, voice):
    """Synthesize text with one voice; returns a float32 numpy
    array. Prefers the resident server (Kokoro already loaded);
    any server failure falls back to the one-shot bridge and the
    server restarts fresh on the next job."""
    try:
        return _speak_via_server(text, voice)
    except Exception as exc:
        print("kokoro server path failed (%r) -- one-shot bridge "
              "fallback" % (exc,))
        _stop_kokoro_server()
        return _speak_one_shot(text, voice)


def generate_audio(job):
    """Generate one audio (TTS) job; returns the final .mp3 path."""
    jid = job["id"]
    text = (job.get("prompt") or "").strip()
    if not text:
        raise ValueError("audio job has no text to speak")
    voice = VOICE_MAP.get(str(job.get("style") or "").strip().lower(),
                          DEFAULT_VOICE)

    set_progress(jid, 10, "generating")
    try:
        samples = _speak(text, voice)
    except Exception as exc:
        print("voice %s failed (%r) — falling back to %s"
              % (voice, exc, DEFAULT_VOICE))
        samples = _speak(text, DEFAULT_VOICE)
    set_progress(jid, 60, "generating")

    wav_path = os.path.join(WORK_DIR, jid + ".wav")
    mp3_path = os.path.join(WORK_DIR, jid + ".mp3")
    sf.write(wav_path, samples, KOKORO_SR)
    cmd = [FFMPEG, "-y", "-i", wav_path, "-codec:a", "libmp3lame",
           "-q:a", "2", mp3_path]
    result = subprocess.run(cmd, capture_output=True, text=True,
                            timeout=300)
    if result.returncode != 0 or not os.path.exists(mp3_path):
        print("ffmpeg mp3 encode failed, writing mp3 via soundfile:",
              result.stderr[-300:] if result.stderr else "")
        sf.write(mp3_path, samples, KOKORO_SR, format="MP3")
    set_progress(jid, 95, "generating")
    return mp3_path


## Chat engine — Qwen3-8B (this worker also chats)

This worker does not only make media — it can also **answer the
TOQIX Agent's chats**. The brain is **Qwen3-8B** (agent maturity
upgrade, prepared 2026-10-15), an AI language model that runs
**right here on this GPU** — loaded in 8-bit it takes roughly
8.5–9 GB of VRAM, which fits a free T4 because the chat loader
frees the video pipeline first. It runs in **non-thinking mode**
for chat: fast, direct answers instead of long reasoning traces.
If it ever fails to load, the worker falls back automatically to
the previous brain, **Qwen2.5-7B-Instruct** (one constant —
`CHAT_MODEL_ID_FALLBACK` — in the code cell below). There is
**no external API**: the conversation never leaves this machine
and the website's own database — nothing is sent to any outside
service.

A chat job's prompt carries the whole conversation as JSON. The
worker feeds it to the model through its chat template and **streams
the reply to the website while it is being written**, so the user
watches the answer appear piece by piece. When the answer is
complete it is posted one final time as final — that final post is
what marks the chat job **done**. A chat job produces no file, so
nothing is imported for it.


In [ ]:
import json
import threading

from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)

# --- Chat model (agent maturity upgrade, prepared 2026-10-15) --------
# THE one line that chooses the agent's brain. Default: Qwen3-8B --
# the strongest Qwen3-class instruct model that fits a free T4
# (16 GB) in 8-bit: weights take ~8.5-9 GB and, because the chat
# loader frees the video pipeline before loading, the rest of the
# card is KV-cache headroom. Rollback is one line: set CHAT_MODEL_ID
# to CHAT_MODEL_ID_FALLBACK (the previous Qwen2.5-7B-Instruct);
# ensure_chat_pipe also drops to the fallback automatically if the
# default model fails to load.
CHAT_MODEL_ID = "Qwen/Qwen3-8B"
CHAT_MODEL_ID_FALLBACK = "Qwen/Qwen2.5-7B-Instruct"
CHAT_MAX_NEW_TOKENS = 512
# Post a partial reply roughly every this many streamed pieces (~tokens).
CHAT_STREAM_EVERY = 30

_chat = {"model": None, "tokenizer": None, "model_id": None}


def ensure_chat_pipe():
    """Lazy-load the chat model on the first chat job and keep it
    resident (8-bit, pinned to the GPU like the video encoder).

    Tries CHAT_MODEL_ID first; if that load fails for any reason
    (a download hiccup, unexpected VRAM pressure), it falls back
    to CHAT_MODEL_ID_FALLBACK so the worker can still chat."""
    if _chat["model"] is None:
        last_exc = None
        for model_id in (CHAT_MODEL_ID, CHAT_MODEL_ID_FALLBACK):
            try:
                print("loading chat model %s (8-bit)..." % model_id)
                tokenizer = AutoTokenizer.from_pretrained(model_id)
                model = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    quantization_config=BitsAndBytesConfig(
                        load_in_8bit=True),
                    device_map={"": "cuda"})
            except Exception as exc:
                print("chat model %s failed to load: %r"
                      % (model_id, exc))
                last_exc = exc
                continue
            _chat["tokenizer"] = tokenizer
            _chat["model"] = model
            _chat["model_id"] = model_id
            print("chat model ready (%s)" % model_id)
            break
        if _chat["model"] is None:
            raise RuntimeError(
                "no chat model could be loaded") from last_exc
    return _chat["model"], _chat["tokenizer"]


def free_chat():
    if _chat["model"] is not None:
        del _chat["model"]
        _chat["model"] = None
    if _chat["tokenizer"] is not None:
        del _chat["tokenizer"]
        _chat["tokenizer"] = None
    _chat["model_id"] = None
    free_cuda()


def _post_reply(job_id, text, final):
    """Post a (partial or final) chat reply to the website.

    Returns True when the website accepted it. Never raises -- a
    website hiccup must not kill a reply that is still being written."""
    try:
        resp = requests.post(
            SITE + "/api/jobs/%s/reply" % job_id,
            json={"text": text, "final": bool(final)},
            timeout=30)
        return resp.ok
    except Exception as exc:
        print("reply post failed (continuing):", repr(exc))
        return False


def _parse_messages(job):
    """A chat job's prompt is a JSON string:
    {"messages": [{"role": ..., "content": ...}, ...]}.

    Defensive: anything unparseable is treated as one plain user
    message, so a malformed job still gets an answer."""
    raw = (job.get("prompt") or "").strip()
    try:
        data = json.loads(raw)
        messages = []
        for item in data["messages"]:
            content = str(item.get("content") or "")
            if content:
                messages.append(
                    {"role": str(item.get("role") or "user"),
                     "content": content})
        if messages:
            return messages
    except Exception:
        pass
    return [{"role": "user", "content": raw}]


def generate_chat(job):
    """Answer one chat job. Returns None: a chat job produces NO
    file -- the reply is streamed to the website as it is generated
    and the final post is what marks the job done (the claim loop
    treats a chat job's None return as DONE and skips the import)."""
    jid = job["id"]
    set_progress(jid, 10, "generating")
    model, tokenizer = ensure_chat_pipe()
    messages = _parse_messages(job)
    # Chat runs in NON-thinking mode: Qwen3 is a hybrid-thinking
    # model, and a thinking trace would stream straight into the
    # user's reply and burn the token budget. enable_thinking=False
    # is the chat template's own switch; templates that do not know
    # it (the Qwen2.5 fallback) may reject it, hence the fallback.
    try:
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except (TypeError, ValueError):
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=CHAT_MAX_NEW_TOKENS,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id)
    thread = threading.Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()
    set_progress(jid, 50, "generating")
    parts = []
    pieces = 0
    for piece in streamer:
        parts.append(piece)
        pieces += 1
        if pieces % CHAT_STREAM_EVERY == 0:
            _post_reply(jid, "".join(parts), False)
    thread.join()
    full = "".join(parts).strip()
    if not _post_reply(jid, full, True):
        # The final post is the job's only completion signal; if it
        # did not land, fail loudly instead of leaving the job stuck.
        raise RuntimeError("final chat reply could not be posted")
    print("[%s] chat job %s answered (%d chars)"
          % (WORKER, jid, len(full)))
    return None


## Run the runner

Leave this cell running. It polls the website every 20 seconds, takes the
oldest waiting job, generates it with the matching engine, and imports the
finished file (importing is what marks the job **done** on the website).
On any error the job is marked **failed** and the loop keeps going; on a
CUDA out-of-memory it first frees the other models and retries once.
Interrupt the cell (Runtime → Interrupt execution) to stop.


In [ ]:
POLL_SECONDS = 10  # idle poll: a fresh prompt is claimed within ~10s
_handled_keys = set()  # dedupe keys handled in this runner session
WAITING_STATUSES = ("queued", "exported")  # "exported" = legacy waiting


def run_job(job):
    kind = job.get("kind") or "video"
    prepare_gpu_for(kind)
    if kind == "image":
        return generate_image(job)
    if kind == "audio":
        return generate_audio(job)
    if kind == "chat":
        return generate_chat(job)
    return generate_video(job)


def free_other_models(kind):
    if kind != "video":
        free_video()
        free_music()
    if kind != "image":
        free_image()
    if kind != "audio":
        free_audio()
    if kind != "chat":
        free_chat()



def free_all_models():
    """Free EVERY resident model (the OOM retry's full reset)."""
    free_video()
    free_image()
    free_chat()
    free_music()
    free_audio()


# --- Resident models (speed build, 2026-10-15) ---------------------
# The ensure_* loaders already keep each engine's model in a
# module-level slot; this registry is the discipline that makes
# residency SAFE on a 16 GB card: a resident model is freed ONLY
# when a different GPU kind is about to run (prepare_gpu_for,
# called at the top of run_job) or when the OOM retry frees
# everything and retries once. Consecutive jobs of the same kind
# reuse the loaded model -- no reload between them. Audio is a
# CPU engine (the Kokoro venv), so it never forces a free: GPU
# residents stay warm across interleaved audio jobs.
_RESIDENT_FREERS = {"video": free_video, "image": free_image,
                    "chat": free_chat}


def resident_kinds():
    """Kinds whose model is currently resident on the GPU."""
    kinds = []
    if _video["pipe"] is not None:
        kinds.append("video")
    if _image["pipe"] is not None:
        kinds.append("image")
    if _chat["model"] is not None:
        kinds.append("chat")
    return kinds


def prepare_gpu_for(kind):
    """Free the OTHER GPU-resident models before a GPU kind runs,
    so the incoming load always has the card to itself. The kind's
    own resident model (if any) is kept -- that is the warm win."""
    if kind not in _RESIDENT_FREERS:
        return  # audio (CPU) and unknown kinds free nothing
    before = resident_kinds()
    for other, freer in _RESIDENT_FREERS.items():
        if other != kind:
            freer()
    if kind != "video":
        free_music()
    if len(before) > (1 if kind in before else 0):
        # Log the switch: this line in the worker log is the proof
        # the kind-switch free actually happened for a given job.
        print("prepare_gpu_for(%s): freed %s -> resident now %s"
              % (kind, [k for k in before if k != kind],
                 resident_kinds()))


def run_job_with_oom_retry(job):
    try:
        return run_job(job)
    except Exception as exc:
        if not is_cuda_oom(exc):
            raise
        print("CUDA out of memory -- freeing ALL models and "
              "retrying once on a clean GPU")
    # The except block has ENDED, so `exc` and its traceback are
    # released. That matters: a traceback pins the failed attempt's
    # stack frames, and the interrupted render call's locals
    # (latents, activations, half-built pipeline state) are
    # referenced by those frames -- while the old code retried from
    # INSIDE the except block, that memory could never be freed and
    # the retry could OOM again on a job a clean card renders
    # easily (self-test video b2f7783c9dc7, 2026-10-15, failed with
    # 10.5 GiB still allocated in-process). The retry below runs
    # after a FULL reset -- every resident model freed, including
    # the failed kind's own -- so attempt 2 starts from the same
    # clean state as a fresh worker's first job of that kind.
    free_all_models()
    free_cuda()
    return run_job(job)


def claim_next(candidates):
    """Atomically claim one candidate job (oldest first) for THIS worker.

    Returns (dedupe_key, claimed_job) for the first claim that wins, or
    None when every candidate is already held by another worker — two
    workers can never generate the same job."""
    for key, cand in candidates:
        ok, claimed = claim_job(cand["id"])
        if ok:
            return key, claimed
        print("[%s] job %s is held by another worker — trying the next"
              % (WORKER, cand["id"]))
    return None


WORKER_CAPS = ["video", "image", "audio", "chat"]


def ping_worker():
    """Tell the website this worker is alive and which job kinds it
    can take. Never raises -- a failed ping must not kill the loop."""
    try:
        requests.post(
            SITE + "/api/workers/ping",
            json={"worker": WORKER, "caps": WORKER_CAPS},
            timeout=15,
        )
    except Exception as exc:
        print("worker ping failed (continuing):", repr(exc))


# --- Claim heartbeat ------------------------------------------------
# Generation can stay silent for a long time (a ~12 GB model download
# posts no step progress), while the site releases any claim whose
# heartbeat is older than 45 minutes. The rebind below records the
# last progress posted per job, and a daemon thread re-posts it every
# 4 minutes while a job runs, so a live worker never loses its claim
# in the middle of a download.
import threading

_orig_set_progress = set_progress
_last_progress = {}


def set_progress(job_id, progress, status, error=None):  # noqa: F811
    _last_progress[job_id] = progress
    return _orig_set_progress(job_id, progress, status, error)


def _heartbeat(job_id, stop_event):
    fails = 0
    while not stop_event.wait(240):
        # The WHOLE beat body is guarded and the thread NEVER exits
        # on a failure: a silently dead heartbeat is how a live
        # worker's job went stale and got re-claimed in a ping-pong
        # loop (job dd92666f4b98, 2026-10-15). Consecutive failures
        # are counted and logged -- and the beating continues no
        # matter how many posts fail in a row.
        try:
            _orig_set_progress(job_id,
                               _last_progress.get(job_id, 1),
                               "generating")
            fails = 0
        except Exception as exc:
            fails += 1
            print("heartbeat post failed x%d (continuing): %r"
                  % (fails, exc))


# --- Fleet idle self-exit --------------------------------------
# GPU quota is precious: a worker that has not claimed a job for
# IDLE_EXIT_SECONDS leaves the session, so an empty queue stops
# burning GPU hours (the fleet manager starts a fresh worker when
# new jobs arrive). The check lives in the poll path only -- a
# generation runs synchronously inside one loop iteration, so
# this can never interrupt a job mid-flight.
IDLE_EXIT_SECONDS = 25 * 60  # 25 min without a claim -> exit
_last_claim_at = time.time()

ping_worker()
print("TOQIX worker %s live — polling %s every %ds. Interrupt to stop."
      % (WORKER, SITE, POLL_SECONDS))
while True:
    try:
        ping_worker()
        job = claim_next_job()
        if job is None:
            print(time.strftime("%H:%M:%S"), "no waiting jobs — sleeping")
            if time.time() - _last_claim_at >= IDLE_EXIT_SECONDS:
                print("[fleet] idle 25m — worker exiting")
                break
        else:
            _last_claim_at = time.time()
            _key = (job.get("kind") or "video",
                    (job.get("prompt") or "").strip().lower(),
                    job.get("aspect"), job.get("duration"),
                    job.get("style"))
            _handled_keys.add(_key)
            jid = job["id"]
            kind = job.get("kind") or "video"
            print("=" * 60)
            print("[%s] claimed job %s (kind=%s): %s"
                  % (WORKER, jid, kind, (job.get("prompt") or "")[:80]))
            started = time.time()
            _hb_stop = threading.Event()
            _hb = threading.Thread(target=_heartbeat,
                                     args=(jid, _hb_stop),
                                     daemon=True)
            _hb.start()
            try:
                out_path = run_job_with_oom_retry(job)
                if kind == "chat":
                    # Chat jobs return None and produce no file:
                    # generate_chat already streamed the reply to the
                    # website and its final post marked the job done
                    # -- there is nothing to import.
                    print("[%s] job %s done in %.0fs (chat reply)"
                          % (WORKER, jid, time.time() - started))
                else:
                    # Stage breadcrumb 99: generation is complete --
                    # the only step left is the import itself (which
                    # is bounded and retried inside import_file).
                    set_progress(jid, 99, "generating")
                    if import_file(jid, out_path):
                        print("[%s] job %s done in %.0fs -> %s"
                              % (WORKER, jid, time.time() - started,
                                 out_path))
                    else:
                        print("[%s] job %s: import failed" % (WORKER, jid))
                        set_progress(jid, 0, "failed", error="output import to the site failed")
            except Exception as exc:
                print("[%s] job %s failed: %r" % (WORKER, jid, exc))
                traceback.print_exc()
                set_progress(jid, 0, "failed", error="%s: %s" % (type(exc).__name__, exc))
            finally:
                _hb_stop.set()
    except Exception as exc:
        print("poll error (will retry):", repr(exc))
    time.sleep(POLL_SECONDS)
